In [7]:
import pandas as pd
from pathlib import Path
import pickle

base = "../data/activemq"

methods      = pd.read_csv(f"{base}/method.csv")
metrics      = pd.read_csv(f"{base}/metrics.csv")
edges        = pd.read_csv(f"{base}/method-invocate-method.csv")
ground_truth = pd.read_csv(f"{base}/ground_truth.csv")
classinfo    = pd.read_csv(f"{base}/classinfo.csv")
methodinfo   = pd.read_csv(f"{base}/methodInfo.csv")

print("=== methods ===")
print(methods.columns.tolist())  # ADDED - see actual column names
print(methods.head())
print(methods.shape)

print("\n=== metrics ===")
print(metrics.columns.tolist())
print(metrics.head())
print(metrics.shape)

print("\n=== edges ===")
print(edges.columns.tolist())  # ADDED - see actual column names
print(edges.head())
print(f"Total edges: {len(edges)}")

print("\n=== classinfo ===")  # ADDED - you never printed this
print(classinfo.columns.tolist())
print(classinfo.head())

print("\n=== methodinfo ===")  # ADDED - you never printed this
print(methodinfo.columns.tolist())
print(methodinfo.head())

print("\n=== ground_truth ===")
print(ground_truth.columns.tolist())  # ADDED
print(ground_truth.head())

# FIXED - use label column directly instead of deriving it
n_smelly = (ground_truth['label'] == 1).sum()
n_total  = len(ground_truth)
print(f"Smelly methods: {n_smelly} / {n_total} ({100*n_smelly/n_total:.2f}%)")

# ADDED - sanity check: is label just source != target?
derived  = (ground_truth['source_class_id'] != ground_truth['target_class_id']).astype(int)
mismatch = (derived != ground_truth['label']).sum()
print(f"Label vs derived mismatch: {mismatch} rows")

# ADDED - check nulls across all files
print("\n=== Null checks ===")
for name, df in [("methods", methods), ("metrics", metrics), 
                  ("edges", edges), ("ground_truth", ground_truth),
                  ("classinfo", classinfo), ("methodinfo", methodinfo)]:
    nulls = df.isnull().sum().sum()
    print(f"{name}: {nulls} total nulls")

# ADDED - check ID consistency across files
print("\n=== ID Consistency ===")
print(f"Unique method IDs in ground_truth: {ground_truth['method_id'].nunique()}")
print(f"Unique method IDs in metrics:      {metrics.columns.tolist()}")
# ^^^ print metrics columns first so you know what the ID column is called

# pkl files
print("\n=== method_tokens.pkl ===")
with open(f"{base}/method_tokens.pkl", "rb") as f:
    method_tokens = pickle.load(f)
print(f"Type: {type(method_tokens)}")

# FIXED - don't assume it's a list, handle dict or list
if isinstance(method_tokens, dict):
    print("Structure: dict")
    print(list(method_tokens.items())[:3])
elif isinstance(method_tokens, list):
    print("Structure: list")
    print(method_tokens[:3])
else:
    print(f"Unexpected type: {type(method_tokens)}")

print("\n=== class_tokens.pkl ===")  # ADDED - you forgot this one
with open(f"{base}/class_tokens.pkl", "rb") as f:
    class_tokens = pickle.load(f)
print(f"Type: {type(class_tokens)}")
if isinstance(class_tokens, dict):
    print(list(class_tokens.items())[:3])
elif isinstance(class_tokens, list):
    print(class_tokens[:3])

=== methods ===
['Unnamed: 0', 'completename', 'nodeId', 'label', 'CC', 'PC', 'LOC', 'NCMIC', 'NCMEC', 'NECA', 'NAMFAEC', 'classID', 'NCM']
   Unnamed: 0                                       completename  nodeId  \
0           0  org_apache_activemq_transport_amqp_client_util...       0   
1           1  org_apache_activemq_broker_QueueSubscriptionTe...       1   
2           2  org_apache_activemq_junit_ActiveMQTestRunner_w...       2   
3           3  org_apache_activemq_transport_amqp_AmqpFramePa...       3   
4           4  org_apache_activemq_transport_amqp_AmqpHeader_...       4   

   label  CC  PC  LOC  NCMIC  NCMEC  NECA  NAMFAEC  classID  NCM  
0      0   1   1    2      1      0     0        0        0    1  
1      0   1   0   11      1      4     2        2        1    5  
2      0   2   2   11      0      1     1        1        2    1  
3      0   1   1    5      1      0     0        0        3    1  
4      0   1   1    3      0      0     0        0        4    0  
(